# Do you and AlexNet find the same things similar?

**Reading:** Mahner, F. P., Muttenthaler, L., Güçlü, U., & Hebart, M. N. (2025). Dimensions underlying the representational alignment of deep neural networks with humans. *Nature Machine Intelligence, 7*, 848–859.

Mahner et al. gave humans and a network the same task: **which of three images is the odd one out?** Today you give this a try with AlexNet.

1. **The images:** 108 images of 36 objects from THINGS, the image database used in the paper
2. **From image to matrix:** filters, feature maps (conv1) and the activation matrix (fc7)
3. **Odd One Out:** you and AlexNet pick the odd one out

**How to use this notebook: settings in CAPITALS (e.g. `EXAMPLE`) can be changed. Change them and run the cell again.**

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

**If Sciebo doesn't work:** download the Week 6 folder `objects` from Sciebo, and upload it into a folder called `images` inside this notebook's folder (`Cognitive_Neuroscience_Update/Week6-Mahner2025`).

In [2]:
# Find the course data: on JupyterHub it is in Sciebo, on your own computer next to this notebook
hits = sorted(Path.home().glob("sciebo/*/Week6/images/objects"))
IMAGE_DIR = hits[0] if hits else Path("images/objects")
if not IMAGE_DIR.is_dir() or not any(IMAGE_DIR.iterdir()):
    raise FileNotFoundError(
        f"Images were not found in {IMAGE_DIR}.\n"
        "On JupyterHub: did you connect Sciebo when starting the server, and accept the shared CNU folder?\n"
        "On your own computer: put the folder at images/objects"
    )

# Folders: objects/<category>/<object>/<image>. Each image gets a short name, e.g. "flamingo_2"
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
images, category = {}, {}
for cat_dir in sorted(p for p in IMAGE_DIR.iterdir() if p.is_dir()):
    for obj_dir in sorted(p for p in cat_dir.iterdir() if p.is_dir()):
        files = sorted(f for f in obj_dir.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"} and not f.name.startswith("."))
        for i, f in enumerate(files, 1):
            name = f"{obj_dir.name}_{i}"
            images[name] = what_network_sees(Image.open(f).convert("RGB"))
            category[name] = cat_dir.name
names = list(images)
print(f"{len(images)} images in {len(set(category.values()))} categories")

108 images in 9 categories


In [3]:
# The trained AlexNet, as in the last weeks
trained = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).to(device).eval()

normalise = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

LAYERS = {
    "conv1": ("features", 1), "conv2": ("features", 4), "conv3": ("features", 7),
    "conv4": ("features", 9), "conv5": ("features", 11),
    "fc6": ("classifier", 2), "fc7": ("classifier", 5),       # fc7 = the second-to-last layer
}

def get_acts(model, imgs):
    # Activations of each layer. Conv layers: (images, filters, height, width); fc layers: (images, units)
    saved = {}
    hooks = [getattr(model, part)[i].register_forward_hook(
                 lambda layer, inp, out, name=name: saved.__setitem__(name, out.detach()))
             for name, (part, i) in LAYERS.items()]
    with torch.no_grad():
        model(torch.stack([normalise(img) for img in imgs]).to(device))
    for h in hooks:
        h.remove()
    return {layer: a.numpy() for layer, a in saved.items()}

def get_features(model, imgs):
    # One row per image and layer. Conv layers: each feature map averaged over its positions, as in the paper
    acts = get_acts(model, imgs)
    return {layer: a.mean((2, 3)) if a.ndim == 4 else a for layer, a in acts.items()}

features = get_features(trained, [images[n] for n in names])
row = {n: i for i, n in enumerate(names)}

## 1. The images

108 images: 9 categories, 4 objects per category, 3 images per object.

Use this table to get the names of the images you want to use to build your own triplets in section 3.
Take a moment to think about which triplets AlexNet might answer differently from you, and which ones you would both answer the same way.

In [ ]:
cats = list(dict.fromkeys(category.values()))
per_row = max(sum(category[n] == c for n in names) for c in cats)
fig, axes = plt.subplots(len(cats), per_row, figsize=(1.5 * per_row, 1.75 * len(cats)))
for ax in axes.ravel():
    ax.axis("off")
for r, c in enumerate(cats):
    for k, n in enumerate([n for n in names if category[n] == c]):
        axes[r, k].imshow(images[n]); axes[r, k].set_title(n, fontsize=8)
    axes[r, 0].text(-0.15, 0.5, c.replace("_", " "), transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
plt.tight_layout(); plt.show()

In [23]:
SEED = None        # type any number, e.g. 7 or 2026

if SEED is None:
    raise ValueError("Please type a number for SEED (any number you like) and run this cell again.")
rng = np.random.default_rng(SEED)
cat_a, cat_b = rng.choice(sorted(set(category.values())), 2, replace=False)
objects_of = lambda c: sorted({n.rsplit("_", 1)[0] for n in names if category[n] == c})
two = rng.choice(objects_of(cat_a), 2, replace=False)               # two objects of one category
one = rng.choice(objects_of(cat_b))                                 # one object of another
EXAMPLE = [f"{o}_{rng.integers(1, 4)}" for o in [*two, one]]        # one random image of each (1 to 3)
rng.shuffle(EXAMPLE)
print("Your triplet:", EXAMPLE)

Your triplet: ['vase_1', 'lemon_1', 'banana_3']


## 3. Odd One Out

Build your own triplets from the names in section 1. For each, write down which image **you** think is the odd one out (1, 2 or 3), then see what AlexNet chooses.

#### Q1. Before running section 3, build a couple of triplets where you expect AlexNet to agree with you and a couple where you expect it to disagree, and note why. Were you right? Relate what you see to the comparison of humans and the network in the article presented today.

In [ ]:
MY_TRIPLETS = [
    (EXAMPLE, None),                                     # your random triplet from section 2: replace None with your odd one out (1, 2 or 3)
    (["flamingo_2", "koala_3", "hairdryer_1"], 3),       # example: your odd one out is image 3
    # (["name_1", "name_2", "name_3"], your answer),
]
LAYER = "fc7"      # the layer AlexNet uses for its choice (as in the paper: fc7). Try conv1 ... fc6 too

def odd_one_out(triplet, layer="fc7"):
    z = features[layer][[row[n] for n in triplet]]
    a, b = max([(0, 1), (0, 2), (1, 2)], key=lambda p: z[p[0]] @ z[p[1]])
    return 3 - a - b + 1                       # position (1, 2 or 3) of the image not in the pair

if any(answer is None for _, answer in MY_TRIPLETS):
    raise ValueError("Please write your odd one out (1, 2 or 3) for every triplet, including your random triplet.")

fig, axes = plt.subplots(len(MY_TRIPLETS), 3, figsize=(7, 2.5 * len(MY_TRIPLETS)), squeeze=False)
for t, (triplet, mine) in enumerate(MY_TRIPLETS):
    alexnet = odd_one_out(triplet, layer=LAYER)
    for k, name in enumerate(triplet):
        ax = axes[t, k]; ax.imshow(images[name]); ax.set_xticks([]); ax.set_yticks([])
        labels = (["you"] if mine == k + 1 else []) + (["AlexNet"] if alexnet == k + 1 else [])
        ax.set_title(" + ".join(labels), fontsize=10, color="black" if len(labels) == 2 else "tab:red")
        ax.set_xlabel(name, fontsize=8)
        for side in ax.spines.values():
            side.set_linewidth(3 if labels else 0); side.set_color("black" if len(labels) == 2 else "tab:red")
plt.tight_layout(); plt.show()